# JobShopRL: Sistema de RL para Job Shop Scheduling

Este notebook muestra cómo utilizar el sistema modular JobShopRL en Google Colab.

## 1. Instalación y configuración

Primero, vamos a clonar el repositorio (o crear la estructura de directorios y archivos)

In [ ]:
# Opción 1: Si el código está en GitHub
# !git clone https://github.com/username/jobshop_rl.git

# Opción 2: Crear estructura de directorios manualmente
!mkdir -p jobshop_rl/{models,environment,agents,rewards,heuristics,utils,experiments,data}/{training_problems,test_problems}
!mkdir -p results/plots logs

Instalamos las dependencias necesarias:

In [ ]:
!pip install -q torch numpy matplotlib pandas

## 2. Generar problemas de ejemplo

Primero, vamos a generar algunos problemas aleatorios para usar como ejemplos:

In [ ]:
import sys
sys.path.append('/content')  # Asegúrate de que la ruta esté en el path de Python

from jobshop_rl.data.problem_loader import ProblemLoader
import os

# Crear directorios para los problemas
os.makedirs('jobshop_rl/data/training_problems', exist_ok=True)
os.makedirs('jobshop_rl/data/test_problems', exist_ok=True)

# Generar problemas de entrenamiento (más pequeños)
for i in range(3):
    problem = ProblemLoader.generate_random_problem(
        num_jobs=6, 
        num_machines=6,
        seed=42+i
    )
    file_path = f"jobshop_rl/data/training_problems/problem_train_{i + 1}.json"
    ProblemLoader.save_problem(problem, file_path, format='json')
    print(f"Problema de entrenamiento {i+1} guardado en {file_path}")

# Generar problemas de prueba (más grandes)
for i in range(2):
    problem = ProblemLoader.generate_random_problem(
        num_jobs=8, 
        num_machines=8,
        seed=100+i
    )
    file_path = f"jobshop_rl/data/test_problems/problem_test_{i + 1}.json"
    ProblemLoader.save_problem(problem, file_path, format='json')
    print(f"Problema de prueba {i+1} guardado en {file_path}")

## 3. Experimento con un único problema (FT10)

Ahora, vamos a ejecutar un experimento con el problema clásico FT10:

In [ ]:
from jobshop_rl.experiments.factory import ExperimentFactory

# Configurar parámetros
reward_params = {
    "makespan_weight": 1.0,
    "idle_weight": 0.2,
    "critical_weight": 0.1,
    "balance_weight": 0.05,
    "progress_weight": 0.2,
    "local_improvement_weight": 0.15
}

agent_params = {
    "lr": 0.0003,
    "gamma": 0.99,
    "entropy_coef": 0.02,
    "K_epochs": 4,
    "use_lr_decay": True,
    "use_grad_clip": True,
    "advantage_normalization": True,
    "gae_lambda": 0.95,
    "seed": 42
}

# Ejecutar experimento con menos episodios para que sea más rápido
agent, results = ExperimentFactory.run_full_experiment(
    episodes=50,  # Reducido para que sea más rápido en Colab
    reward_strategy="advanced",
    agent_params=agent_params,
    reward_params=reward_params,
    visualize=True,
    save_plots=True,
    csv_logging=True,
    csv_filename="ft10_training.csv",
    csv_base_dir="./"
)

### Visualizamos los resultados

In [ ]:
from IPython.display import display

# Mostrar la mejor planificación encontrada
print(f"Mejor makespan: {agent.best_makespan}")
display(results['plots']['best_schedule'])

# Mostrar evolución del makespan durante el entrenamiento
display(results['plots']['training_makespan'])

# Comparación con heurísticas
print("\nComparación con heurísticas:")
for heuristic, improvement in results['comparison'].items():
    print(f"vs {heuristic}: {improvement:.2f}% de mejora")

## 4. Experimentación por lotes con múltiples problemas

Ahora, vamos a ejecutar un experimento por lotes con los problemas generados:

In [ ]:
from jobshop_rl.experiments.batch_experimenter import BatchExperimenter

# Configurar experimentador por lotes
experimenter = BatchExperimenter(
    training_dir="jobshop_rl/data/training_problems",
    test_dir="jobshop_rl/data/test_problems",
    output_dir="./results",
    agent_params=agent_params,
    reward_strategy="advanced",
    reward_params=reward_params,
    seed=42
)

# Entrenar el agente con pocos episodios para demo
best_agent = experimenter.train_agent(episodes_per_problem=20)

### Evaluar el mejor agente en los problemas de prueba

In [ ]:
results = experimenter.evaluate_on_test_set(best_agent)

# Mostrar resultados
display(results)

# Mostrar visualizaciones
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Encontrar imágenes de soluciones
solution_images = glob.glob('./results/plots/test/*.png')
for img_path in solution_images:
    plt.figure(figsize=(12, 6))
    img = mpimg.imread(img_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title(img_path.split('/')[-1])
    plt.show()

## 5. Descargar resultados

Finalmente, podemos descargar los resultados generados:

In [ ]:
# Comprimir los resultados
!zip -r jobshop_rl_results.zip ./results ./logs

# Descargar el archivo ZIP
from google.colab import files
files.download('jobshop_rl_results.zip')